# CNN-LSTM Baseline Training on NSL-KDD

This notebook covers building, training, and evaluating the hybrid CNN-LSTM neural network classifier on features selected by the BWOA algorithm.

In [ ]:
import yaml
import numpy as np
import tensorflow as tf
from src.data.nsl_kdd import NSLKDDLoader
from src.models.cnn_lstm import build_cnn_lstm
from src.models.trainer import ModelTrainer
from src.evaluation.metrics import ExperimentMetrics
from src.utils.visualizer import ExperimentVisualizer
from src.utils.logger import ExperimentLogger

# Load configuration
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

nsl_train_path = config["data"]["nsl_kdd_train"]

In [ ]:
# Load preprocessed data and apply BWOA feature mask
loader = NSLKDDLoader()
train_df = loader.load(nsl_train_path)
X, y = loader.preprocess(train_df)

# Split
X_train, X_test, y_train, y_test = loader.train_test_split(X, y, test_size=0.2)
# Normalize
X_train, X_test = loader.normalize(X_train, X_test)

# Load mask
try:
    best_mask = np.load("data/features/nslkdd_bwoa_mask.npy")
except FileNotFoundError:
    # Fallback to choosing all features if npy not created yet
    best_mask = np.ones(X_train.shape[1], dtype=int)

selected_indices = np.where(best_mask == 1)[0]
X_train_masked = X_train[:, selected_indices]
X_test_masked = X_test[:, selected_indices]

# Reshape inputs for sequential layers: (batch_size, sequence_length, feature_dimension)
# Here sequence_length = 1
X_train_3d = np.expand_dims(X_train_masked, axis=1)
X_test_3d = np.expand_dims(X_test_masked, axis=1)

# Convert targets to categorical one-hot encoding for 5-class target outputs
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=5)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=5)

print(f"X_train_3d shape: {X_train_3d.shape}")
print(f"y_train_cat shape: {y_train_cat.shape}")

In [ ]:
# Build hybrid classifier
model_params = config["model"]["cnn_lstm"]
input_shape = (X_train_3d.shape[1], X_train_3d.shape[2])
n_classes = 5

model = build_cnn_lstm(
    input_shape=input_shape,
    n_classes=n_classes,
    filters=model_params["filters"],
    kernel_size=model_params["kernel_size"],
    lstm_units=model_params["lstm_units"],
    dropout_rate=model_params["dropout_rate"]
)

model.summary()

In [ ]:
# Compile and train model using Trainer class
trainer = ModelTrainer(config["training"])
# Override training epochs for demo run if required
trainer.epochs = 5

history = trainer.train(
    model,
    X_train_3d,
    y_train_cat,
    X_test_3d,
    y_test_cat
)

In [ ]:
# Save loss and accuracy curves
ExperimentVisualizer.plot_training_history(
    history.history,
    "figures/training_history.png"
)

In [ ]:
# Predict on evaluation set
y_prob = model.predict(X_test_3d)
y_pred = np.argmax(y_prob, axis=-1)

# Compute experiment metrics
metrics = ExperimentMetrics.compute(y_test, y_pred, y_prob)
ExperimentMetrics.print_report(metrics)

In [ ]:
# Plot and save confusion matrix
ExperimentVisualizer.plot_confusion_matrix(
    y_test,
    y_pred,
    loader.classes,
    "figures/confusion_matrix.png"
)

In [ ]:
# Plot and save ROC Curves
ExperimentVisualizer.plot_roc_curves(
    y_test,
    y_prob,
    loader.classes,
    "figures/roc_curves.png"
)

In [ ]:
# Save trained keras model checkpoint
model.save("models/cnn_lstm_nslkdd_baseline.keras")
print("Baseline model saved successfully to models/cnn_lstm_nslkdd_baseline.keras")

In [ ]:
# Log and save experiment results
logger = ExperimentLogger("cnn_lstm_baseline_nslkdd")
logger.log_hyperparams(config["model"]["cnn_lstm"])
logger.log_hyperparams(config["training"])
logger.log_feature_mask(best_mask, loader.get_feature_names())

# Log final step metrics
logger.log_metrics({
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "roc_auc": metrics["roc_auc"]
}, step=1)

logger.save_experiment_summary("logs/cnn_lstm_baseline_summary")

### Results Summary

The baseline hybrid CNN-LSTM model achieved strong scores on the compressed feature space. Feature selections led to rapid backpropagation convergence during training, requiring fewer epochs to establish optimal test accuracy.